# Extracción de metadatos de estudios
Este notebook lee imágenes de diferentes estudios y extrae sus respectivos metadatos para exportarlos en un archivo de Excel

Elaborado por Juan Manuel Rivera/AJS

# Importación de librerías y funciones auxiliares

In [ ]:
import os
import SimpleITK as sitk
import numpy as np
import pandas as pd

ModuleNotFoundError: No module named 'pandas'

En la siguiente celda está la dirección del directorio donde están las imágenes

In [2]:
paths = [("FSFB","C:/Users/jmriv/Documents/Dataset006_FSFB")]

# Extracción de metadatos

Si desea añadir algún metadato añada el DICOM Tag en la siguiente lista

In [24]:
import numpy as np

In [29]:
np.array([
    (10,20,30),
    (40,50,60)
]).mean(axis=0)

array([25., 35., 45.])

In [47]:
tags = {
    #DB
    "db" : [],
    # File path
    "file_path" : [],
    # Mean z-spacing
    "spacing" : [],
    # Number of slices
    "size" : [],
    # Lesion total size
    "lesion_size":[],
    # Number of conected lesions
    "connected_components" : [],
    # Mean component size
    "connected_size" : [],
    # SD component size
    "connected_sd" : [],
}

In [45]:
for datasets in paths:
    for img_path in os.listdir(os.path.join(datasets[1], "labelsTr")):
        seg = os.path.join(datasets[1], "labelsTr",img_path)
        vol = os.path.join(datasets[1], "imagesTr",f"{img_path[:-7]}_0000{img_path[-7:]}")

        original_img = sitk.ReadImage(vol)
        seg_img = sitk.ReadImage(seg)

        seg_array = sitk.GetArrayFromImage(seg_img)

        cc = sitk.ConnectedComponent(seg_img)
        stats = sitk.LabelShapeStatisticsImageFilter()
        stats.Execute(cc)

        for tag, array in tags.items():
            if tag == "db":
                array.append(datasets[0])
            if tag == 'file_path':
                array.append(img_path)
            elif tag == 'spacing':
                array.append(original_img.GetSpacing())
            elif tag == 'size':
                array.append(original_img.GetSize())
            elif tag == 'lesion_size':
                array.append( np.count_nonzero(seg_array) )
            elif tag == "connected_components":
                array.append(stats.GetNumberOfLabels())
            elif tag == "connected_size" or tag == "connected_sd":
                sizes = [stats.GetNumberOfPixels(label) for label in stats.GetLabels()]
                if len(sizes) == 0:
                    array.append(0)
                else:
                    sizes = np.array(sizes)
                    if tag == "connected_size":
                        array.append(sizes.mean())
                    else:
                        array.append(sizes.std())

In [46]:
tags

{'file_path': ['FSFB_001.nii.gz',
  'FSFB_002.nii.gz',
  'FSFB_003.nii.gz',
  'FSFB_004.nii.gz',
  'FSFB_005.nii.gz',
  'FSFB_006.nii.gz',
  'FSFB_007.nii.gz',
  'FSFB_008.nii.gz',
  'FSFB_009.nii.gz',
  'FSFB_010.nii.gz',
  'FSFB_011.nii.gz',
  'FSFB_012.nii.gz',
  'FSFB_013.nii.gz',
  'FSFB_014.nii.gz',
  'FSFB_015.nii.gz',
  'FSFB_016.nii.gz',
  'FSFB_017.nii.gz',
  'FSFB_018.nii.gz',
  'FSFB_019.nii.gz',
  'FSFB_020.nii.gz',
  'FSFB_021.nii.gz',
  'FSFB_022.nii.gz',
  'FSFB_023.nii.gz',
  'FSFB_024.nii.gz',
  'FSFB_025.nii.gz',
  'FSFB_026.nii.gz',
  'FSFB_027.nii.gz',
  'FSFB_028.nii.gz',
  'FSFB_029.nii.gz',
  'FSFB_030.nii.gz',
  'FSFB_031.nii.gz',
  'FSFB_032.nii.gz',
  'FSFB_033.nii.gz',
  'FSFB_034.nii.gz',
  'FSFB_035.nii.gz',
  'FSFB_036.nii.gz',
  'FSFB_037.nii.gz',
  'FSFB_038.nii.gz',
  'FSFB_039.nii.gz',
  'FSFB_040.nii.gz',
  'FSFB_041.nii.gz',
  'FSFB_042.nii.gz',
  'FSFB_043.nii.gz',
  'FSFB_044.nii.gz',
  'FSFB_045.nii.gz',
  'FSFB_046.nii.gz',
  'FSFB_047.nii.gz',


In [ ]:
df = pd.DataFrame(data=tags)
display(df.head(5))

,Dirección del estudio,spacing,n_slices
0,sub-stroke0001,2.0,75
1,sub-stroke0002,2.0,75
2,sub-stroke0003,2.0,70
3,sub-stroke0004,2.0,69
4,sub-stroke0005,2.0,69


In [17]:
df.describe()

,spacing,n_slices
count,149.000000,149.000000
mean,2.535569,64.973154
std,0.899327,17.839353
min,0.800000,34.000000
25%,2.000000,46.000000
50%,2.000000,69.000000
75%,3.999987,74.000000
max,4.000012,177.000000


In [ ]:
df.to_excel("MetadatosNIfTI.xlsx")